# Data Cleaning: Impacts of Data Centers on Electrical Utilities

This notebook loads the raw data from `data/raw/`.

In [ ]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
DATA_DIR = Path("data/raw")

# display options for pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

## First look at each dataset

### FracTracker U.S. Data Centers

In [ ]:
data_centers = pd.read_csv(DATA_DIR / "fractracker_data_centers.csv")
print("Rows and columns:", data_centers.shape)
data_centers.sample(3)

In [ ]:
# Summary Statistics for Data Centers Dataset
# Looking for anything that needs to be cleaned
data_centers.describe(include="all").T

### FracTracker U.S. Data Centers - Data Cleaning

| Feature | Identified Issue | Proposed Solution |
|---|---|---|
| `mw` | Contains both numerical values and ranges. | Separate ranges into `mw_low` and `mw_high` to preserve the available information. |
| `power_source` | String categorical values. | Standardize categories and encode them for modeling. |
| `dedicated_power_plant` | Limited availability across facilities. | Determine if power plants can be identified using location (latitude/longitude) and power source. |
| `number_of_generators` | Contains numerical information stored as strings; limited availability across facilities. | Too limited to be a learnable feature unless more information can be added. |
| `number_of_buildings` | Contains integers, ranges, and strings (e.g., `6+`, `1-19`, `8`, `3 large, 1 small`). | Create a consistent approach for representing ranges and descriptive counts. |
| `cooling_source` | Categorical values stored as strings. | Standardize categories and encode them for modeling. |
| `cooling_type` | Inconsistent categorical formatting (e.g., `Closed loop` vs. `Closed Loop`). | Normalize categorical values and encode them for modeling. |
| `project_cost` | Contains strings and potentially ranges rather than consistent numerical values. | Extract numerical values and determine an appropriate method for representing ranges. |



`mw`

In [ ]:
# Remove commas and spaces
mw = data_centers['mw'].str.replace(',', '').str.strip()

# Split and assign ranges
data_centers[['mw_low', 'mw_high']] = mw.str.split('-', expand=True)
data_centers['mw_high'] = data_centers['mw_high'].fillna(data_centers['mw_low']) # fill where no range exists 

# Convert to numeric
data_centers[['mw_low', 'mw_high']] = (
    data_centers[['mw_low', 'mw_high']]
    .apply(pd.to_numeric, errors='coerce')
)

data_centers.loc[data_centers['mw'].notna()][['mw', 'mw_low', 'mw_high']].head()

`power_source`

In [ ]:
# Normalize power sources
power_source = (
    data_centers['power_source']
    .str.lower()
    .str.strip()
    .str.replace('(', '')
    .str.replace(')', '')
    .str.replace('/', '_')
    .str.replace(' ', '_')
)

# One-hot encoding for each source type
source_encoded = power_source.str.get_dummies(sep=",_")
data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = [
    "power_source", "batteries", "biomass", "coal", "fuel_cells", 
    "geothermal", "grid_unspecified_mix", "hybrid_onsite_and_grid", 
    "hydroelectric", "natural_gas", "nuclear", "oil_diesel", 
    "renewable", "solar", "wind"
]

data_centers.loc[data_centers['power_source'].notna()][columns].sample(5)

`number_of_buildings`

In [ ]:
# Standardization function for assigning number of buildings
def standardize_buildings(buildings):
    if pd.isna(buildings):
        return np.nan, np.nan
    
    buildings = buildings.replace(' ', '')
    b_low = np.nan
    b_high = np.nan
    
    if '-' in buildings:
        split = buildings.split('-')
        b_low = int(split[0])
        b_high = int(split[1])
        
    elif '+' in buildings:
        buildings = buildings.replace('+', '')
        b_low = int(buildings)
        
    elif buildings.isdigit():
        b_low = int(buildings)
        b_high = int(buildings)
        
    return b_low, b_high
    
# Add new features to handle ranges
data_centers[["buildings_low", "buildings_high"]] = pd.DataFrame(
    data_centers["number_of_buildings"].apply(standardize_buildings).tolist(),
    index=data_centers.index
)

data_centers.loc[data_centers["buildings_low"].notna()][["number_of_buildings", "buildings_low", "buildings_high"]].head()

`cooling_source` + `cooling_type`

In [ ]:
for col in ['cooling_source', 'cooling_type']:
    # Normalize cooling cols
    cooling_data = (
        data_centers[col]
        .str.lower()
        .str.strip()
        .str.replace('/', '_')
        .str.replace(' ', '_')
    )

    # One-hot encoding for each source type
    source_encoded = cooling_data.str.get_dummies()
    data_centers = pd.concat([data_centers, source_encoded], axis=1)

columns = ['cooling_source', 'air', 'water', 'hybrid_air_water']

data_centers.loc[data_centers['cooling_source'].notna()][columns].sample(5)

In [ ]:
columns = ['cooling_type', 'closed_loop', 'fans', 'open_loop']

data_centers.loc[data_centers['cooling_type'].notna()][columns].sample(5)

`project_cost`

In [ ]:
def standardize_project_cost(cost):
    if pd.isna(cost):
        return np.nan, np.nan
    
    multiplier = 1
    cost_low = np.nan
    cost_high = np.nan
    
    # Normalize cost
    cost = (
        str(cost)
        .lower()
        .replace(' ', '')
        .replace('$', '')
        .replace(',', '')
        .replace('miillion', 'million')
    )

    # Replace abbreviated m
    cost = re.sub(r'm$', 'million', cost)
    
    if 'million' in cost:
        multiplier = 1000000
    elif 'billion' in cost:
        multiplier = 1000000000
        
    cost = re.sub(r'[mb]illion', '', cost)
    
    # Assign range to low and high
    if '-' in cost:
        split = cost.split('-')
        cost_low = float(split[0]) * multiplier
        cost_high = float(split[1]) * multiplier
        
    elif '+' in cost:
        cost = cost.replace('+', '')
        cost_low = float(cost) * multiplier
        
    else:
        cost_low = float(cost) * multiplier
        cost_high = float(cost) * multiplier
        
    return cost_low, cost_high

# Add new features for handling the cost range
data_centers[["project_cost_low", "project_cost_high"]] = pd.DataFrame(
    data_centers["project_cost"].apply(standardize_project_cost).tolist(),
    index=data_centers.index
)

data_centers[data_centers["project_cost"].notna()][["project_cost", "project_cost_low", "project_cost_high"]].sample(5)

### FracTracker Crypto Mining Operations

In [ ]:
crypto = pd.read_csv(DATA_DIR / "fractracker_crypto.csv")
print("Rows and columns:", crypto.shape)
crypto.sample(3)

In [ ]:
crypto.describe(include="all").T

### EIA-860 Plant Data

The first row of each EIA-860 Excel file is a title, so the column names are on the second row (`header=1`).

In [ ]:
# Added header=1 to skip the first row

plants = pd.read_excel(DATA_DIR / "eia860_2025" / "2___Plant_Y2025.xlsx", header=1)
print("Rows and columns:", plants.shape)
plants.sample(3)

In [ ]:
plants.describe(include="all").T

### EIA-860 Generator Data

The generator file has three sheets: `Operable`, `Proposed`, and `Retired and Canceled`. For now we only look at `Operable`.

In [ ]:
generators = pd.read_excel(
    DATA_DIR / "eia860_2025" / "3_1_Generator_Y2025.xlsx",
    sheet_name="Operable",
    header=1
)
print("Rows and columns:", generators.shape)
generators.sample(3)

In [ ]:
generators.describe(include="all").T

### EIA-861M Retail Sales of Electricity


In [ ]:
# Added header=2 to skip the first two rows of the Excel sheet

sales = pd.read_excel(DATA_DIR / "eia861m_sales_revenue.xlsx", sheet_name="Monthly-States", header=2)
print("Rows and columns:", sales.shape)
sales.sample(3)

In [ ]:
sales.describe(include="all").T

### EIA-930 Balance Data

In [ ]:
# This is a subset of the EIA 930 Balance dataset for the second half of 2025

balance = pd.read_csv(DATA_DIR / "eia930" / "EIA930_BALANCE_2025_Jul_Dec.csv")
print("Rows and columns:", balance.shape)
balance.sample(3)

In [ ]:
balance.describe(include="all").T

### EIA-930 Subregion Data

In [ ]:
subregion = pd.read_csv(DATA_DIR / "eia930" / "EIA930_SUBREGION_2025_Jul_Dec.csv")
print("Rows and columns:", subregion.shape)
subregion.sample(3)

In [ ]:
subregion.describe(include="all").T

## Save cleaned data

Save the cleaned data centers to `data/processed/` so later notebooks can load them without re-running this one.

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

data_centers.to_csv(PROCESSED_DIR / "fractracker_data_centers_clean.csv", index=False)
print("Saved: fractracker_data_centers_clean.csv with", data_centers.shape[0], "rows and", data_centers.shape[1], "columns")